# 01 · Data pipeline (FEVER + SciFact)

**Run with:** Runtime → Change runtime type → **CPU, High-RAM** (no GPU needed), then Runtime → **Run all**.

What it does, all cached in Google Drive under `MyDrive/neural-evidence/` so later notebooks and reruns skip finished steps:
1. Downloads FEVER 1.0 (train, labelled dev, Wikipedia dump) and SciFact.
2. Builds a SQLite store of the Wikipedia dump and a BM25 page index (slowest step, roughly 30–60 min, done once).
3. Writes `train/val/test.jsonl` for FEVER (val split grouped by claim text, BM25 evidence for NEI) and `test.jsonl` for SciFact.
4. Prints statistics and samples, and saves `data_report.txt` to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/neural-evidence'
CODE = f'{BASE}/code'          # upload the code folder here, or set REPO_URL
REPO_URL = ''                   # e.g. 'https://github.com/<you>/<repo>.git' once the repo exists
RAW, WIKI, PROC = f'{BASE}/data/raw', f'{BASE}/data/wiki', f'{BASE}/data/processed'
LOCAL = '/content/work'         # fast local disk for the heavy build
for d in (RAW, WIKI, PROC, LOCAL): os.makedirs(d, exist_ok=True)

In [ ]:
if not os.path.exists(f'{CODE}/ne') and REPO_URL:
    !git clone -q {REPO_URL} /content/code
    CODE = '/content/code'
assert os.path.exists(f'{CODE}/ne'), f'Code not found: put the neural-evidence folder at {CODE} or set REPO_URL'
%cd {CODE}
!pip -q install -r requirements.txt pytest
!python -m pytest tests -q

In [ ]:
# 1. Download (skips files already in Drive)
!bash scripts/download.sh {RAW}

In [ ]:
# 2. Wikipedia store + BM25 index: build on local disk, cache in Drive
import shutil, time
db_drive, idx_drive = f'{WIKI}/wiki.db', f'{WIKI}/bm25_index'
db_local, idx_local = f'{LOCAL}/wiki.db', f'{LOCAL}/bm25_index'
if os.path.exists(f'{idx_drive}/page_ids.json'):
    print('Using cached wiki store/index from Drive')
    shutil.copy(db_drive, db_local); shutil.copytree(idx_drive, idx_local, dirs_exist_ok=True)
else:
    t = time.time()
    !python scripts/build_wiki.py --wiki {RAW}/fever/wiki-pages.zip --db {db_local} --index {idx_local}
    print(f'built in {(time.time()-t)/60:.1f} min; caching to Drive')
    shutil.copy(db_local, db_drive); shutil.copytree(idx_local, idx_drive, dirs_exist_ok=True)

In [ ]:
# 3. Processed datasets
!python scripts/prepare_fever.py --raw {RAW}/fever --db {db_local} --index {idx_local} --out {PROC}/fever
!python scripts/prepare_scifact.py --raw {RAW}/scifact --out {PROC}/scifact

In [ ]:
# 4. Report: label balance, evidence lengths, samples
import json, collections, statistics, io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    for ds, split in [('fever','train'),('fever','val'),('fever','test'),('scifact','test')]:
        rows = [json.loads(l) for l in open(f'{PROC}/{ds}/{split}.jsonl')]
        c = collections.Counter(r['label_name'] for r in rows)
        print(f'== {ds}/{split}: {len(rows)} examples', {k: f'{v} ({v/len(rows):.1%})' for k, v in c.items()})
        for lab in c:
            L = [len(r['evidence_gold'].split()) for r in rows if r['label_name'] == lab]
            print(f'   {lab:<16} gold-evidence words: median {statistics.median(L)}, mean {statistics.mean(L):.1f}')
        if 'text_in_train' in rows[0]:
            print('   test claims whose text also appears in train:', sum(r['text_in_train'] for r in rows))
        for lab in c:
            r = next(r for r in rows if r['label_name'] == lab)
            print(f'   [{lab}] {r["claim"]}\n      gold: {r["evidence_gold"][:200]}\n      retr: {r["evidence_retrieved"][:200]}')
    print('== fever stats.json'); print(open(f'{PROC}/fever/stats.json').read())
print(buf.getvalue())
open(f'{BASE}/data_report.txt', 'w').write(buf.getvalue())
print('Saved', f'{BASE}/data_report.txt')